In [ ]:
import random
import time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

SEED = 42
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 256
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5
device = "cuda"

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

df = pd.read_csv("deceptive-opinion.csv")
texts = df["text"].tolist()
labels = (df["deceptive"] == "deceptive").astype(int).values
groups = df["hotel"].values
folds = list(GroupKFold(n_splits=5).split(texts, labels, groups))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode(text_list):
    enc = tokenizer(text_list, truncation=True, max_length=MAX_LEN,
                    padding="max_length", return_tensors="pt")
    return enc["input_ids"], enc["attention_mask"]

def false_positive_rate(y_true, y_pred):
    # share of truthful reviews wrongly flagged as deceptive
    truthful = (y_true == 0)
    return float((y_pred[truthful] == 1).mean())

def run_distilbert(train_idx, test_idx):
    set_seed(SEED)
    X_train = [texts[i] for i in train_idx]
    X_test = [texts[i] for i in test_idx]
    y_train = labels[train_idx]
    y_test = labels[test_idx]

    tr_ids, tr_mask = encode(X_train)
    te_ids, te_mask = encode(X_test)
    train_ds = TensorDataset(tr_ids, tr_mask, torch.tensor(y_train))
    test_ds = TensorDataset(te_ids, te_mask, torch.tensor(y_test))

    gen = torch.Generator()
    gen.manual_seed(SEED)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=gen)
    test_loader = DataLoader(test_ds, batch_size=64)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

    for epoch in range(EPOCHS):
        model.train()
        for ids, mask, lab in train_loader:
            out = model(input_ids=ids.to(device), attention_mask=mask.to(device), labels=lab.to(device))
            out.loss.backward()
            optimizer.step()
            optimizer.zero_grad()

    model.eval()
    preds = []
    with torch.no_grad():
        for ids, mask, lab in test_loader:
            out = model(input_ids=ids.to(device), attention_mask=mask.to(device))
            preds.extend(out.logits.argmax(dim=-1).cpu().tolist())

    del model, optimizer
    torch.cuda.empty_cache()
    return np.array(preds)

def run_tfidf(clf, train_idx, test_idx):
    pipe = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2), clf)
    pipe.fit([texts[i] for i in train_idx], labels[train_idx])
    return pipe.predict([texts[i] for i in test_idx])

rows = []
pred_rows = []
start = time.time()

for k, (train_idx, test_idx) in enumerate(folds):
    y_test = labels[test_idx]
    fold_preds = {
        "tfidf_logreg": run_tfidf(LogisticRegression(max_iter=1000), train_idx, test_idx),
        "tfidf_svm": run_tfidf(LinearSVC(), train_idx, test_idx),
        "distilbert": run_distilbert(train_idx, test_idx),
    }
    for name, pred in fold_preds.items():
        rows.append({
            "fold": k,
            "model": name,
            "macro_f1": f1_score(y_test, pred, average="macro"),
            "false_positive_rate": false_positive_rate(y_test, pred),
            "test_hotels": ",".join(sorted(set(groups[test_idx]))),
        })
        for i, p in zip(test_idx, pred):
            pred_rows.append({"row_id": int(i), "fold": k, "model": name, "pred": int(p)})
    print("fold", k, "done, elapsed (s):", round(time.time() - start))

results = pd.DataFrame(rows)
results.to_csv("results_5fold.csv", index=False)
pd.DataFrame(pred_rows).to_csv("predictions_5fold.csv", index=False)

print()
print(results.pivot(index="fold", columns="model", values="macro_f1").round(3))
print()
summary = results.groupby("model")[["macro_f1", "false_positive_rate"]].agg(["mean", "std"]).round(3)
print(summary)